# Lab 7.8 &mdash; Retry & Idempotency

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Retry a flaky order lookup, but cap the attempts so it can't loop forever
- Make sending idempotent with an order+draft key, so a re-run is safe
- See why idempotency matters most for money & messages

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, and the pipeline logic &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Reliability: retry & idempotency](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-07-08"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Models and tools are flaky, so wrap calls in a **retry** &mdash; but **cap** the attempts (deck slide
12). And design for **idempotency**: key each action so running the same task twice is **safe** and
never **double-sends** an email or **double-charges** a card. This is the subtlest and most important
discipline for anything that touches **money or messages**.

In [ ]:
# A flaky order lookup: the network hiccups n_fail times, then returns the order.
import hashlib
def flaky_lookup(order_id, n_fail):
    calls = {"n": 0}
    def f():
        calls["n"] += 1
        if calls["n"] <= n_fail:
            raise RuntimeError("transient network error")
        return {"id": order_id, "status": "shipped", "eta": "Friday"}
    return f
def raises(fn):
    try: fn(); return False
    except Exception: return True
print("helpers ready: flaky_lookup(id, n) fails n times, then returns the order dict")

## Your Turn
Complete `with_retry` (capped), `send_key` (the idempotency key from the order id + a draft hash),
and `send_once` (idempotent via the key set).

In [ ]:
import hashlib

def with_retry(fn, max_attempts=3):
    # call fn(); on failure retry up to max_attempts; raise the last error if all fail
    last = None
    for attempt in range(max_attempts):
        try:
            return ___   # TODO: call fn() and return its result
        except Exception as e:
            last = e
    raise ___   # TODO: re-raise the last error after the cap is hit

def send_key(order_id, draft):
    # the idempotency key ties the EXACT draft to its order -- re-sending the same one is a no-op
    h = hashlib.sha256(draft.encode()).hexdigest()[:8]
    return ___   # TODO: combine the order id and the draft hash h into one key string

def send_once(key, sent):
    # idempotent: sending the same key twice must NOT double-send
    if ___:   # TODO: this key was already sent
        return "already_sent"
    sent.add(key)
    return "sent"

try:
    print("first try ok   :", with_retry(flaky_lookup("4471", 0))["status"])
    print("after 2 fails  :", with_retry(flaky_lookup("4471", 2), 3)["status"])
    print("exhausted raises:", raises(lambda: with_retry(flaky_lookup("4471", 5), 3)))
    k = send_key("4471", "Hi Priya, your order 4471 is due Friday.")
    print("send key        :", k)
    sent = set()
    print("send (1st)      :", send_once(k, sent))
    print("send (2nd)      :", send_once(k, sent))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("returns the order when the lookup succeeds", lambda: with_retry(flaky_lookup("4471", 0))["status"] == "shipped")
expect_true("succeeds after transient failures", lambda: with_retry(flaky_lookup("4471", 2), 3)["status"] == "shipped")
expect_true("raises once attempts are exhausted", lambda: raises(lambda: with_retry(flaky_lookup("4471", 5), 3)))
expect_true("the send key ties the draft to its order id", lambda: send_key("4471", "hello").startswith("4471"))
expect_true("a different draft yields a different key", lambda: send_key("4471", "hello") != send_key("4471", "goodbye"))
expect_true("the first send goes out", lambda: send_once(send_key("4471", "hi"), set()) == "sent")
expect_true("re-sending the SAME order+draft is suppressed (idempotent)", lambda: (lambda s: (send_once(send_key("4471","hi"), s), send_once(send_key("4471","hi"), s))[1])(set()) == "already_sent")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Retry with a cap, and key every irreversible action so a re-run is safe. Assume every step can fail -- and make failure safe and visible. Idempotency is what lets an automation run unattended.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>